# Layer-3 judge — qwen3.6-35b-a3b judges gemma-4-26b-a4b, gpt-oss-20b, granite-4.2-8b, ministral-3-14b
Launch title: **`bangla-pi-judge-qwen`**. Research plan §4.4 (judge gate) / §10.3 (cross-vendor), DECISIONS D23, D38, **D39**.
Rubric `scoring/judge.py::JUDGE_PROMPT` (unchanged; every record stores its SHA-256). No self-judging: the judge's
vendor (`alibaba`) differs from every agent's; `check_cross_vendor` raises otherwise.

| agent model | attacked status-ok episodes | §4.4-gated |
|---|---|---|
| gemma-4-26b-a4b | 599 | 30 |
| gpt-oss-20b | 599 | 32 |
| granite-4.2-8b | 595 | 58 |
| ministral-3-14b | 599 | 35 |

**Order:** all gated episodes first (pre-registered; they feed judge-augmented ASR), then — with `SCOPE='all'`
(default) — every other attacked episode as a calibration set (L2-vs-judge agreement; never added to ASR).

**Serve:** `ggml-org/Qwen3.6-35B-A3B-GGUF` → `Qwen3.6-35B-A3B-Q4_K_M.gguf` @ `baec3ebee244` (SHA-256 `671e47e0ec53c665…` asserted), exactly the
main-grid flags (Q4_K_M; 1 server tensor-split 1,1 (D17 layout); thinking OFF per plan §11.3 O2. D17 passed no template kwargs, i.e. ran the template default (thinking ON), so 171.4 eps/h is a thinking-ON rate (D30)), llama.cpp `b11344`, `--temp 0 --seed 20261002`, thinking off (template checked).

**GPU estimate** (from the main grid's measured cold-prefill p10 665 tok/s and decode
59.5 tok/s; conservative token count): gated only ≈ **0.49 h**; all ≈
**2.12 h** (worst case, every reply hits 256 tokens: 4.31 h), incl.
0.35 h setup. One session, well under the 11 h deadline guard.

**Self-contained:** cell 1 unpacks the embedded harness + pre-rendered judge inputs (xz tarball 421,848 B,
13 files, sha256 `2f1903551d7dffbe6013cd0edb2cfa2d884ac8010984b55cc7e359fc00fb229d`; inputs sha256 `5e94a7f1e2fa0403…`) and verifies every file.
Inputs are `render_trace()` text of each trace (tool calls + final answer, tool results excluded, D23) + the trace's SHA-256.

**Outputs (/kaggle/working):** `judge/judge_qwen.jsonl` (one verdict per episode; skip-if-done),
`judge/judge_manifest_qwen.json`, `judge_qwen_outputs.zip`, `server_8099.log`, final `JUDGE_RESULT …` line.
**Resume:** attach this version's Output as an input and Save & Run All again.

Settings: Accelerator **GPU T4 x2**, Internet **on**, notebook **private**, then **Save Version → Save & Run All (Commit)**.
Generated by `scripts/build_judge_notebook.py`; prepared 2026-10-07, **not run**.

In [ ]:
# ===== 1. Unpack the embedded judge harness + inputs, verify every sha256 (fails loudly) =====
# Self-contained: no Kaggle dataset or git clone. Built by scripts/build_judge_notebook.py.
import base64, hashlib, io, json, os, shutil, sys, tarfile, time
from pathlib import Path
SESSION_T0 = time.time()
CODE_DIR = '/kaggle/working/code'
JUDGE_KEY = 'qwen'
JUDGE_ID = 'qwen_kaggle'
N_INPUTS, N_GATED = 2392, 155
EMBED_SHA256 = '2f1903551d7dffbe6013cd0edb2cfa2d884ac8010984b55cc7e359fc00fb229d'
EMBED_BYTES = 421848
EMBED_B64 = '''
...
AABJaPG2FBc7MAMAAAAABFla
'''

_blob = base64.b64decode(''.join(EMBED_B64.split()))
EMBED_SHA256_GOT = hashlib.sha256(_blob).hexdigest()
print(f'EMBED tarball: {len(_blob)} B, sha256 {EMBED_SHA256_GOT}', flush=True)
if len(_blob) != EMBED_BYTES or EMBED_SHA256_GOT != EMBED_SHA256:
    raise RuntimeError(f'EMBEDDED TARBALL CORRUPT: got {len(_blob)} B sha256 {EMBED_SHA256_GOT}, '
                       f'expected {EMBED_BYTES} B sha256 {EMBED_SHA256}. Re-import the notebook.')
if os.path.isdir(CODE_DIR):
    shutil.rmtree(CODE_DIR)
os.makedirs(CODE_DIR)
with tarfile.open(fileobj=io.BytesIO(_blob), mode='r:xz') as _tf:
    _members = _tf.getmembers()
    for _m in _members:
        if not _m.isfile() or _m.name.startswith('/') or '..' in _m.name.split('/'):
            raise RuntimeError(f'unsafe tar member {_m.name!r}')
    if hasattr(tarfile, 'data_filter'):
        _tf.extractall(CODE_DIR, members=_members, filter='data')
    else:
        _tf.extractall(CODE_DIR, members=_members)
_root = Path(CODE_DIR)
EMBED_MANIFEST = json.loads((_root / 'EMBED_MANIFEST.json').read_text(encoding='utf-8'))
_bad = [r for r, m in EMBED_MANIFEST['files'].items()
        if not (_root / r).is_file() or hashlib.sha256((_root / r).read_bytes()).hexdigest() != m['sha256']]
_extra = [str(p.relative_to(_root)) for p in _root.rglob('*') if p.is_file()
          and str(p.relative_to(_root)) not in EMBED_MANIFEST['files'] and p.name != 'EMBED_MANIFEST.json']
if _bad or _extra:
    raise RuntimeError(f'EMBED VERIFY FAILED: bad={_bad} extra={_extra}')
INPUTS_PATH = str(_root / 'judge' / 'judge_inputs.jsonl')
CONFIG_PATH = str(_root / 'judge' / 'judge_config.json')
_inputs = [json.loads(l) for l in open(INPUTS_PATH, encoding='utf-8') if l.strip()]
assert len(_inputs) == N_INPUTS and sum(r['gated'] for r in _inputs) == N_GATED, 'input counts differ from header'
assert {r['judge_id'] for r in _inputs} == {JUDGE_ID}, 'inputs routed to another judge'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print(f"verified {len(EMBED_MANIFEST['files'])} files; {N_INPUTS} judge inputs ({N_GATED} gated) for {JUDGE_ID}", flush=True)


In [ ]:
# ===== 2. PARAMETERS =====
SCOPE      = 'all'     # 'all' = gated first, then every other attacked episode (calibration); 'gated' = §4.4 gate only
DEADLINE_H = 11.0      # no NEW judge call after this many hours since cell 1 (Kaggle kills at 12 h)
SMOKE_N    = None      # smoke test only: judge the first N (gated first); None = full pass
WORK       = '/kaggle/working'
OUT_DIR    = '/kaggle/working/judge'
INPUT_ROOT = '/kaggle/input'   # attach a previous version's Output -> its judge_*.jsonl is resumed (skip-if-done)
EST_SESSION_H = {'gated': 0.49, 'all': 2.12, 'all_worst': 4.31}

from runners.main_grid import MODELS, LLAMA_BUILD, SEED
from scoring.judge import RUBRIC_SHA256, load_judge_config, check_cross_vendor
MODEL_KEY = 'qwen'
M = MODELS[MODEL_KEY]
CFG = load_judge_config(CONFIG_PATH)
assert CFG['judges'][JUDGE_ID]['vendor'] != '' and JUDGE_ID in CFG['judges']
for _m in sorted({r['model'] for r in _inputs}):
    check_cross_vendor(JUDGE_ID, _m, CFG)      # raises on any same-vendor pairing
RUN_ID = f'judge_{JUDGE_KEY}'
OUT_JSONL = f'{OUT_DIR}/{RUN_ID}.jsonl'
print(f"{RUN_ID}: judge {M['label']} ({CFG['judges'][JUDGE_ID]['vendor']}) on agents "
      f"{sorted({r['model'] for r in _inputs})}; scope={SCOPE} rubric={RUBRIC_SHA256[:12]}", flush=True)


In [ ]:
# ===== 3. GPU check (needs Accelerator = GPU T4 x2) =====
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout, flush=True)
_n = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.count('GPU ')
print(f'GPUs visible: {_n}', flush=True)
assert _n >= 2, 'Need Accelerator = GPU T4 x2'


In [ ]:
# ===== 4. Pinned llama.cpp (CUDA 12.8 build, same as the main grid) =====
import glob
LLAMA_DIR = '/kaggle/tmp/llama' if os.path.isdir('/kaggle/tmp') else '/tmp/llama'
os.makedirs(LLAMA_DIR, exist_ok=True)
_rel = f'https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}'
for _url in (f'{_rel}/cudart-llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz',
             f'{_rel}/llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz'):
    _fn = f"{LLAMA_DIR}/{_url.split('/')[-1]}"
    subprocess.run(['wget', '-q', '-O', _fn, _url], check=True)
    subprocess.run(['tar', 'xzf', _fn, '-C', LLAMA_DIR], check=True)
SERVER = glob.glob(f'{LLAMA_DIR}/**/llama-server', recursive=True)[0]
print('llama-server at', SERVER, flush=True)
print(subprocess.run([SERVER, '--version'], capture_output=True, text=True).stderr[:300], flush=True)
import httpx, pandas  # noqa: F401  (preinstalled on Kaggle; the judge client needs httpx)


In [ ]:
# ===== 5. Pinned GGUF -> scratch (never /kaggle/working, D13); assert size + SHA-256 =====
from huggingface_hub import hf_hub_download
for _p in glob.glob('/kaggle/working/models/**/*', recursive=True):
    if _p.endswith(('.gguf', '.incomplete', '.lock')) and os.path.isfile(_p):
        os.remove(_p)
MODELS_DIR = '/kaggle/tmp/models' if os.path.isdir('/kaggle/tmp') else '/tmp/models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['HF_HUB_CACHE'] = os.environ['HUGGINGFACE_HUB_CACHE'] = MODELS_DIR
_free = shutil.disk_usage(MODELS_DIR).free / 2**30
print(f'MODELS_DIR={MODELS_DIR} free={_free:.1f} GiB', flush=True)
assert _free > M['min_free_gib'], f"scratch too small for {M['file']}"
_t0 = time.time()
MODEL_PATH = hf_hub_download(M['repo'], M['file'], revision=M['rev'], local_dir=MODELS_DIR)
assert os.path.getsize(MODEL_PATH) == M['size'], f'size mismatch {os.path.getsize(MODEL_PATH)} != {M["size"]}'
_h = hashlib.sha256()
with open(MODEL_PATH, 'rb') as _f:
    while (_b := _f.read(1 << 22)):
        _h.update(_b)
GGUF_SHA = _h.hexdigest()
print(f"{M['file']} {M['size']} B sha256 {GGUF_SHA} match={GGUF_SHA == M['sha256']} ({time.time()-_t0:.0f}s)", flush=True)
assert GGUF_SHA == M['sha256'], 'GGUF sha256 mismatch'


In [ ]:
# ===== 6. Serve in the layout + variant that passed the pilot; prove template kwargs =====
import requests
from runners.main_grid import server_commands, template_ok
PROCS = []
def _wait(p, port, timeout_s=900):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if p.poll() is not None:
            return False
        try:
            if requests.get(f'http://127.0.0.1:{port}/v1/models', timeout=5).ok:
                return True
        except Exception:
            pass
        time.sleep(3)
    return False
def _template_tail(port):
    try:
        r = requests.post(f'http://127.0.0.1:{port}/apply-template', timeout=30,
                          json={'messages': [{'role': 'user', 'content': 'ping'}]})
        return r.json().get('prompt', '')
    except Exception as e:
        return f'<apply-template failed: {e}>'
SPECS = server_commands(M, SERVER, MODEL_PATH, SEED, log_dir=WORK)
for _s in SPECS:
    print(f"serve[{M['variant']}] gpu={_s['gpu']} port={_s['port']}: " + ' '.join(_s['cmd'][1:]), flush=True)
    _t0 = time.time()
    _p = subprocess.Popen(_s['cmd'], env=dict(os.environ, **_s['env']), stdout=open(_s['log'], 'w'),
                          stderr=subprocess.STDOUT)
    PROCS.append(_p)
    if not _wait(_p, _s['port']):
        for _q in PROCS:
            _q.kill()
        raise RuntimeError(f"llama-server on :{_s['port']} failed (rc={_p.poll()}):\n" + open(_s['log']).read()[-2000:])
    print(f"  up on :{_s['port']} ({time.time()-_t0:.0f}s)", flush=True)
PORTS = [_s['port'] for _s in SPECS]
TEMPLATE_TAILS = {p: _template_tail(p) for p in PORTS}
for _p, _t in TEMPLATE_TAILS.items():
    print(f'template[{_p}] tail: {_t[-200:]!r}', flush=True)
TEMPLATE_OK = all(template_ok(M, t) for t in TEMPLATE_TAILS.values())
print(f"TEMPLATE_OK={TEMPLATE_OK} expect={M['template_expect']!r} forbid={M['template_forbid']!r}", flush=True)
if not TEMPLATE_OK:
    for _q in PROCS:
        _q.kill()
    raise RuntimeError('chat-template kwargs did not take effect (thinking/reasoning setting) - not running the grid')
MODEL_ID = requests.get(f'http://127.0.0.1:{PORTS[0]}/v1/models', timeout=10).json()['data'][0]['id']
SERVE_USED = {'layout': M['layout'], 'variant': M['variant'], 'ports': PORTS, 'model_id': MODEL_ID,
              'servers': [{'port': s['port'], 'gpu': s['gpu'], 'cmd': s['cmd'][1:], 'log': s['log']} for s in SPECS],
              'template_ok': TEMPLATE_OK, 'template_tails': {str(k): v[-300:] for k, v in TEMPLATE_TAILS.items()}}
print(f"server up SERVE_VARIANT={M['variant']} layout={M['layout']} ports={PORTS} model_id={MODEL_ID}", flush=True)
print(subprocess.run(['nvidia-smi','--query-gpu=index,memory.used,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout, flush=True)


In [ ]:
# ===== 7. Judge: gated episodes first, then (SCOPE='all') the rest; resumable; deadline guard =====
import glob
from scoring import run_judge
os.makedirs(OUT_DIR, exist_ok=True)
SEEDED = None
if not os.path.exists(OUT_JSONL):
    for _p in sorted(glob.glob(f'{INPUT_ROOT}/**/{RUN_ID}.jsonl', recursive=True)):
        shutil.copy(_p, OUT_JSONL)
        SEEDED = _p
        break
print('resume seed:', SEEDED, flush=True)
CFG['judges'][JUDGE_ID]['base_url'] = f'http://127.0.0.1:{PORTS[0]}/v1'
_cfg_used = f'{OUT_DIR}/judge_config_used.json'
Path(_cfg_used).write_text(json.dumps(CFG, indent=1), encoding='utf-8')
_args = ['--config', _cfg_used, '--inputs', INPUTS_PATH, '--out', OUT_JSONL, '--only-judge-id', JUDGE_ID,
         '--gated-first', '--deadline-ts', str(SESSION_T0 + DEADLINE_H * 3600)]
if SCOPE == 'all':
    _args.append('--ungated')
if SMOKE_N:
    _args += ['--limit', str(SMOKE_N)]
_t0 = time.time()
JUDGE_SUMMARY = run_judge.main(_args)
JUDGE_WALL_S = time.time() - _t0
print('JUDGE_SUMMARY', json.dumps(JUDGE_SUMMARY), f'wall {JUDGE_WALL_S:.0f}s', flush=True)


In [ ]:
# ===== 8. Stop server, tally, manifest, JUDGE_RESULT line =====
import collections, datetime as _dt, zipfile
for _p in PROCS:
    try:
        _p.terminate()
    except Exception:
        pass
_keys = ['model', 'defense', 'suite', 'user_task', 'injection_task', 'condition', 'rep']
_done = {}
for _l in open(OUT_JSONL, encoding='utf-8'):
    if _l.strip():
        _r = json.loads(_l)
        _done[tuple(_r[k] for k in _keys)] = _r
_want_g = {tuple(r[k] for k in _keys) for r in _inputs if r['gated']}
_want_a = {tuple(r[k] for k in _keys) for r in _inputs}
TALLY = collections.Counter((r['model'], r['condition'], 'gated' if r['gated'] else 'ungated', r['verdict'])
                            for r in _done.values())
_unparse = sum(1 for r in _done.values() if str(r.get('reason', '')).startswith(('unparseable', 'judge parse error')))
_trunc = sum(1 for r in _done.values() if r.get('finish_reason') == 'length')
_lat = sorted(r.get('latency_s') or 0 for r in _done.values())
SUMMARY = {'judged': len(_done), 'gated_done': len(_want_g & set(_done)), 'gated_total': len(_want_g),
           'all_done': len(_want_a & set(_done)), 'all_total': len(_want_a),
           'complete_gated': _want_g <= set(_done), 'complete_all': _want_a <= set(_done),
           'verdicts': dict(collections.Counter(r['verdict'] for r in _done.values())),
           'full_on_gated': sum(1 for r in _done.values() if r['gated'] and r['verdict'] == 'full'),
           'unparseable': _unparse, 'finish_length': _trunc,
           'latency_s_median': _lat[len(_lat)//2] if _lat else None}
WALL_S = time.time() - SESSION_T0
MANIFEST = {'run_id': RUN_ID, 'judge_id': JUDGE_ID, 'judge_model': M['label'], 'scope': SCOPE, 'smoke_n': SMOKE_N,
            'gguf': {k: M[k] for k in ('repo', 'rev', 'file', 'size')} | {'sha256': GGUF_SHA},
            'serve': SERVE_USED, 'llama_build': LLAMA_BUILD, 'seed': SEED, 'rubric_sha256': RUBRIC_SHA256,
            'embed': {'sha256': EMBED_SHA256, 'bytes': EMBED_BYTES},
            'inputs_sha256': EMBED_MANIFEST['files']['judge/judge_inputs.jsonl']['sha256'],
            'resume_seeded': SEEDED, 'run_judge_summary': JUDGE_SUMMARY, 'summary': SUMMARY,
            'tally': [list(k) + [v] for k, v in sorted(TALLY.items())], 'est_session_h': EST_SESSION_H,
            'session_started': _dt.datetime.fromtimestamp(SESSION_T0).isoformat(timespec='seconds'),
            'session_ended': _dt.datetime.now().isoformat(timespec='seconds'), 'wall_s': round(WALL_S, 1)}
MANIFEST_PATH = f'{OUT_DIR}/judge_manifest_{JUDGE_KEY}.json'
Path(MANIFEST_PATH).write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
with zipfile.ZipFile(f'{WORK}/{RUN_ID}_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as _z:
    for _f in (OUT_JSONL, MANIFEST_PATH, f'{OUT_DIR}/judge_config_used.json') + tuple(s['log'] for s in SERVE_USED['servers']):
        if os.path.exists(_f):
            _z.write(_f, os.path.basename(_f))
print(json.dumps(SUMMARY, indent=1), flush=True)
if not SUMMARY['complete_gated'] or (SCOPE == 'all' and not SUMMARY['complete_all']):
    print('INCOMPLETE: attach this version\'s Output as an input and Save & Run All again (skip-if-done).', flush=True)
print(f"JUDGE_RESULT {RUN_ID} judge={M['label']} gated={SUMMARY['gated_done']}/{SUMMARY['gated_total']} "
      f"all={SUMMARY['all_done']}/{SUMMARY['all_total']} full_on_gated={SUMMARY['full_on_gated']} "
      f"verdicts={json.dumps(SUMMARY['verdicts'], sort_keys=True)} unparseable={_unparse} "
      f"wall_h={WALL_S/3600:.2f} embed={EMBED_SHA256[:12]}", flush=True)


## After the run (on the box)
Download `judge_qwen_outputs.zip`, check the `JUDGE_RESULT` line and that `embed` matches this header, put
`judge_qwen.jsonl` under `code/results/judge/`, then
`python -m analysis.run_analysis ... --judge 'results/judge/judge_*.jsonl'` (judge-augmented ASR uses gated verdicts
only, D39). Record slug/version, wall time and `JUDGE_RESULT` in `DECISIONS.md`.